# Simulate linearly mixed pixels from satellite sensors

This notebook imports an spectral library and water spectra taken from imagery to then simulate pixels as would be seen by satellite sensors. The number of pixels is user-defined, as are the classes allowed to be simulated into each pixel. For each class, the range of potential fractional percent cover (FPC) and the presence probability are set to default values but can be altered by the user if desired.

### Set up the notebook

In [1]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import linear_mixing as lmx

# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

In [66]:
# Plotting settings and function
plt.rcParams['font.family'] = 'serif' #sans-serif'

colours = {
    "kelp": "gold",
    "brown_algae": "saddlebrown" ,
    "red_veg": "r",
    "green_veg": "palegreen",
    "mineral" : "grey",
    "water" : "blue",
    "other" : "grey",
    
    "macrocystis": "goldenrod",
    "ecklonia": "gold" ,
    "undaria" : "orange",
    
    "durvillaea": "mediumpurple",
    'phyllospora' : "darkmagenta", 
    'cystophora': "mediumslateblue", 
    'carpophyllum': "rebeccapurple", 
    'scytosiphon': "blueviolet", 
    'petalonia': "plum", 
    'acrocarpia': "orchid", 
    'hormosira': "violet", 
    'sargassum': "purple", 
    
    'frondose_rhodophyte': "indianred", 
    'filamentous_rhodophyte': "firebrick", 
    'rhodophyte' : "maroon",
    
    'ulva': "limegreen", 
    'grass': "green", 
    
    'mussels': "thistle", 
    'worm_castings': "slategray", 
    'barnacle_shells': "darkseagreen", 
    'shell_litter': "lightsteelblue", 
    'sand': "silver", 
    'gravel': "darkgrey", 
    'rock': "dimgrey", 
}

def plot_mixpix_components(pixel_fpcs: pd.DataFrame, colours: dict, sensor_code: str, class_code: str, save = False, output_dir = None, n_cols = 2):
    """ 
    Plot the fractional percent cover distributions of the simulated pixels.
    """   
    fpcs = pixel_fpcs.copy()
    fpcs["count"] = fpcs.astype(bool).sum(axis=1)

    mat_count = fpcs.shape[1]
    fig_rows = mat_count // n_cols if mat_count %n_cols== 0 else mat_count // n_cols + 1
    
    # Create a figure and subplot axes
    fig, ax = plt.subplots(fig_rows, n_cols,
        figsize=(10, fig_rows * 2),
        sharey=True
    )
    fig.tight_layout(w_pad = -0.2, h_pad= -0.5)

    # Sublot loop
    for col in range(mat_count):
        col_name = fpcs.columns[col]
        row_i, col_i = divmod(col, n_cols)
        
        ax[row_i, col_i].hist(fpcs.iloc[:,col], 
                               bins = 50,
                               range = (0, max(fpcs.iloc[:, col])), 
                               color = colours.get(col_name, "navy"), log = True)
        
        annotation = col_name.replace("_", "\n").capitalize()
        ax[row_i, col_i].annotate(annotation,  
                                   xy = (0.2, 0.7), 
                                   xycoords = "axes fraction")
    for row in range(fig_rows-1):
        for col in range(n_cols):
            ax[row, col].set_xticklabels([])
    
    for column in range (n_cols):
        if fpcs.columns[col] != "count":
            ax[-1, column].set_xlabel("Fractional Cover")
        else:
            ax[-1, column].set_xlabel("Number of Classes Simulated")
            
    for row in range (fig_rows):
        ax[row, 0].set_ylabel("Pixel Count")

    # Hide last empty subplot if odd number of columns
    if mat_count % fig_rows != 0:
        empties = n_cols - (mat_count % n_cols) +1
        for empty in range(1, empties):
            ax[-1, -empty].set_visible(False)
            ax[-2, -empty].set_xlabel("Fractional Cover", fontsize = 10)
            ax[-2, -empty].set_xticks([0, 0.5, 1])
            ax[-2, -empty].set_xticklabels(["0", "0.5", "1"])
            
    if save: 
        if not sensor_code or not class_code:
            ValueError("Sensor code and class code must be provided to save the figure.")
        if output_dir is None :
            output_dir = "./"
        
        os.makedirs(output_dir, exist_ok=True)
        filename = f"mixpix_comps_dist.svg"
        filepath = os.path.join(output_dir, filename)
        fig.savefig(filepath, bbox_inches = "tight")
    plt.show()

## User-definable parameters

General settings: directories, pixel counts, sensor choice, class scheme choice

In [33]:
# Which sensors and class scheme do you want to use?
sensor_code = "S2"
class_code = "Class"

max_classes = 7 if class_code == "Class" else None

# How many pixels do you want?
dev_pixel_count = 500
unseen_pixel_count = 100

# Do you want to save your plots, and if so, where?
save_plots = False
plot_output_dir = f"C:/Users/s4770224/Documents/Work/Writing/Figures/Obj2/water/{sensor_code}/{class_code}/"
os.makedirs(plot_output_dir, exist_ok= True)

# Where should simulation results be saved?
sim_pix_directory = f"./data/mixed_sims/test/{sensor_code}/{class_code}/"
os.makedirs(sim_pix_directory, exist_ok= True)

# Where are your spectra?
if sensor_code.lower() == "s2":
    spec_lib_path = "./data/processed/resampled/mine_and_specchio/sub-1-spectra_no-standardization/noisy_Sentinel_2_ABC_resampled.csv"
elif sensor_code.lower() == "enm":
    spec_lib_path = "./data/processed/resampled/mine_and_specchio/sub-1-spectra_no-standardization/noisy_Enmap_resampled.csv"
elif sensor_code.lower() == "psd":
        spec_lib_path = "./data/processed/resampled/mine_and_specchio/sub-1-spectra_no-standardization/noisy_Superdove_resampled.csv"
else: 
    raise ValueError("Incorrect sensor code. Pick from S2, enm or psd.")

labels_path = "data/feb2026_combo_labels_prepped.csv"

Now define the granular settings: Which genera should be counted as which classes? Which classes do you want to simulate? How much of each class is allowed in a pixel? How likely is it that a class may be found in a pixel? 

Alternatively, if your grouping is already defined, just run the next cell. 

In [34]:
# choose settings depending on class code

if class_code.lower() == "kbrgm":
    # Settings for KBRGM
    # Which classes are being simulated? What genera are in each?
    material_options = {
        "kelp" : ['ecklonia', 
                'macrocystis', 
                'undaria'],
        "brown_algae" : ['acrocarpia', 
                'cystophora', 
                'carpophyllum', 
                'durvillaea', 
                'hormosira', 
                'petalonia',
                'phyllospora', 
                'sargassum', 
                'scytosiphon'],
        "red_veg" : ['frondose_rhodophyte',
                    'filamentous_rhodophyte',
                    "rhodophyte"], 
        "green_veg" : ["ulva", "grass"],
        "mineral" : ['barnacle_shells', 
                    'gravel', 
                    'mussels', 
                    'rock', 
                    'sand', 
                    'shell_litter', 
                    'worm_castings'
                    ],
        }

    # Assign fractional cover ranges to simulate within
    cover_ranges = {
        "kelp": (0,1),
        "brown_algae" : (0, 1),
        "red_veg" : (0,1),
        "green_veg" : (0, 1),
        "mineral" : (0, 1),
        }

    # Assign odds of presence for each class ((1, 0) makes 50/50 odds)
    presence_odds = {
        "kelp" : (3, 1),
        "brown_algae" : (3, 1),
        "red_veg" : (3, 1),
        "green_veg" : (3, 1),
        "mineral" : (3, 1),
        }

    all_allowed = []
    for item in material_options.items():
        all_allowed += item[1]

    labels = pd.read_csv(labels_path, index_col=0)

elif class_code.lower() == "class": 
    #Settings for all materials by class
    labels = pd.read_csv(labels_path, index_col=0)

    all = labels["Class"].unique().tolist()

    material_options = {x:[x] for x in all}
    cover_ranges = {x:(0,1) for x in all}
    presence_odds = {x:(1,0) for x in all}

    all_allowed = []
    for item in material_options.items():
        all_allowed += item[1]

    
elif class_code.lower() == "bo": 
    # Settings for BO (browns detection)
    # Which classes are being simulated? What genera are in each?
    material_options = {
        "brown_algae" : ['ecklonia', 
                        'macrocystis', 
                        'petalonia', 
                        'undaria',
                        'acrocarpia', 
                        'cystophora', 
                        'carpophyllum', 
                        'durvillaea', 
                        'hormosira', 
                        'phyllospora', 
                        'sargassum', 
                        'scytosiphon',
                        ],
        "other" : ['barnacle_shells', 
                'filamentous_rhodophyte',
                'frondose_rhodophyte', 
                'gravel', 
                'grass',
                'mussels',
                'sand', 
                'shell_litter',
                'rock', 
                'ulva', 
                'worm_castings'],
        }

    # Assign fractional cover ranges to simulate within
    cover_ranges = {
        "brown_algae" : (0, 1),
        "other" : (0,1),
        }

    # Assign odds of presence for each class ((1, 0) makes 50/50 odds)
    presence_odds = {
        "brown_algae" : (1, 0),
        "other" : (1, 0),
        }

    all_allowed = []
    for item in material_options.items():
        all_allowed += item[1]
        
    print(f"Classes to be simulated : {list(material_options.keys())} " )

    labels = pd.read_csv(labels_path, index_col=0)
    labels["bo"] = labels["kbrgm"].replace(["red_veg", "green_veg", "mineral"], "other")
    labels["bo"] = labels["bo"].replace(["kelp", "other_brown_alg"], "brown")
    
elif class_code.lower() == "brgm":
    # Settings for BRGM
    # Which classes are being simulated? What genera are in each?
    material_options = {
        "brown_algae" : ['ecklonia', 
                        'macrocystis', 
                        'petalonia', 
                        'undaria',
                        'acrocarpia', 
                        'cystophora', 
                        'carpophyllum', 
                        'durvillaea', 
                        'hormosira', 
                        'phyllospora', 
                        'sargassum', 
                        'scytosiphon',
                        ],
        "red_veg" : ['frondose_rhodophyte',
                    'filamentous_rhodophyte'], 
        "green_veg" : ["ulva", "grass"],
        "mineral" : ['barnacle_shells', 
                    'gravel', 
                    'mussels', 
                    'rock', 
                    'sand', 
                    'shell_litter', 
                    'worm_castings'
                    ],
        }

    # Assign fractional cover ranges to simulate within
    cover_ranges = {
        "brown_algae" : (0, 1),
        "red_veg" : (0,1),
        "green_veg" : (0, 1),
        "mineral" : (0, 1),
        }

    # Assign odds of presence for each class ((1, 0) makes 50/50 odds)
    presence_odds = {
        "brown_algae" : (1, 0),
        "red_veg" : (1, 0),
        "green_veg" : (1, 0),
        "mineral" : (1, 0),
        }



    all_allowed = []
    for item in material_options.items():
        all_allowed += item[1]


    labels = pd.read_csv(labels_path, index_col=0)
    labels["brgm"] = labels["kbrgm"].replace(["kelp", "other_brown_alg"], "brown_algae")
        
else: 
    raise ValueError(f"Class code '{class_code}' not recognized. Please choose from 'kbrgm', 'class', 'bo', or define new one here.")

## Load the endmember and water data

In [ ]:
# Load spectral data from file and prep

data = pd.read_csv(spec_lib_path, index_col=0)
print(f"{len(data)} spectra loaded.")
data = pd.concat([labels["Class"], data], axis = 1)
data = data [data["Class"].isin(all_allowed)]
labels = labels[labels["Class"].isin(all_allowed)]
print(f"{len(data)} spectra kept for simulation.")

### Prepare the water spectra

Inspect the water spectra and clusters at bottom of notebook

In [ ]:
# Load and prepare water pixel spectra from file

if sensor_code =="S2":
    deep_waters = pd.read_csv("../../Work/geospatial_analysis/water_sampling/Sen2/S2_deep_water_agg.csv", index_col=0)
    deep_waters["depth"] = "deep"

    # Load shallow water
    shallow_waters = pd.read_csv("../../Work/geospatial_analysis/water_sampling/Sen2/S2_shallow_agg.csv", index_col=0)
    shallow_waters["depth"] = "shallow"

    # combine all water pixels and separate into metadata and spectra
    all_waters = pd.concat([deep_waters, shallow_waters], 
                    axis = 0, ignore_index= True).reset_index(drop = True)
    water_spec = all_waters.iloc[:, 4:-4]
    water_meta = pd.concat([all_waters.iloc[:, :4], all_waters.iloc[:, -4:]], axis = 1)   
elif sensor_code.lower() == "enm":
    water_spec = pd.read_csv("../../Work/geospatial_analysis/water_sampling/Enmap/Enmap_pixel_samples.csv")
    water_spec.head()

    # get rid of bands outside of endmember spectral range
    water_spec = water_spec.iloc[:, 8:86] 

    # Remove bad data from QGIS export
    a = water_spec.values
    b = (a == a[[0], :]).all(axis=0)
    c = (a == a[:, [0]]).all(axis=1)

    water_spec = water_spec.loc[c==0, b==0]
    water_meta = water_spec.index.to_frame()
elif sensor_code.lower() == "psd":
    # Load and prepare water pixel spectra from file
    water_spec = pd.read_csv("../../Work/geospatial_analysis/water_sampling/planet/planet_pixel_sample_1M.csv")

    # get rid of bands outside of endmember spectral range
    water_spec = water_spec.iloc[:, 1:] 

    # Remove bad data from QGIS export
    a = water_spec.values
    b = (a == a[[0], :]).all(axis=0)
    c = (a == a[:, [0]]).all(axis=1)

    water_spec = water_spec.loc[c==0, b==0]
    water_meta = water_spec.index.to_frame()
else:
    print("Sensor code not recognised. Please use S2, ENM, or PSD.")

print(f"Total of {water_spec.shape[0]} water pixels loaded")
water_spec.columns = data.columns[1:]
# De-duplicate water spectra
water_spec.drop_duplicates(inplace = True)
water_meta = water_meta.loc[water_spec.index]
print(f"Total of {water_spec.shape[0]} unique water pixels")

water_spec = water_spec/10000  # Convert to reflectance

## Separate the endmembers and mix pixels

Two sets of pixels will be made: model development pixels (training/testing) and validation pixels (refered to as "unseen pixels" later)

In [37]:
# Split spectra into training and testing sets
data_train, data_test = train_test_split(data, test_size = 0.3, stratify=labels[class_code])

water_train, water_test = train_test_split(water_spec, test_size = 0.3)
water_meta_train = water_meta.loc[water_train.index]
water_meta_test = water_meta.loc[water_test.index]

# Simulate the development pixels

In [ ]:
# Instantiate the pixel mixer
mixer = lmx.LinearMixing(materials = material_options, 
                         cover_ranges = cover_ranges, 
                         pixels = dev_pixel_count,
                         odds_dict = presence_odds,
                         material_spectra= data_train, 
                         water_spectra = water_train)

# Mix pixels and store results
results, f, endmember_indices = mixer.sim_many_pixels(max_class_count = max_classes)

# Format results and clear old variables
sim_pix, pixel_fpcs, endmember_indices = mixer.format_sim_results(results, f, endmember_indices)
del results, f

# Inspect simulated results
print(f"{sim_pix.shape[0]} simulated pixels with {sim_pix.shape[1]} spectral bands \n")


In [53]:
# Add one mat class + water pixels
more_results, more_f, more_endmember_indices = mixer.sim_single_class_pixels(single_pixel_count= 3000, start_value = dev_pixel_count)

# Add pure water pixels
water_results_df, water_fpcs, water_endmembers = mixer.add_water_only_pixels(water_pixels_count= 10000, start_value= more_results.index[-1]+1)

# Combine with multi-class simulations
endmember_indices = pd.concat([endmember_indices, more_endmember_indices, water_endmembers], axis = 0)
sim_pix = pd.concat([sim_pix, more_results.iloc[:, :-1], water_results_df], axis = 0)
pixel_fpcs = pd.concat([pixel_fpcs, more_f, water_fpcs], axis = 0)


In [ ]:
# Plot the components of the simulated pixels      
plot_mixpix_components(pixel_fpcs, colours, sensor_code= sensor_code, class_code = class_code, save = save_plots, output_dir = plot_output_dir, n_cols=4)

In [12]:
# Export the simulated development pixels
np.save(f"{sim_pix_directory}development_pixels.npy", sim_pix)
np.save(f"{sim_pix_directory}development_pixels_columns.npy", pixel_fpcs.columns, allow_pickle = True)
np.save(f"{sim_pix_directory}development_pixel_fpcs.npy", pixel_fpcs)
np.save(f"{sim_pix_directory}development_pixel_endmembers.npy", endmember_indices)
np.save(f"{sim_pix_directory}data_train_indices.npy", data_train.index)

# Simulate the unseen pixels

In [ ]:
# Instantiate the pixel mixer
unseen_mixer = lmx.LinearMixing(materials = material_options, 
                         cover_ranges = cover_ranges, 
                         pixels = unseen_pixel_count, 
                         material_spectra= data_test,
                         odds_dict= presence_odds, # None,
                         water_spectra = water_test)

# Mix pixels and store results
new_results, new_f, new_endmember_indices = unseen_mixer.sim_many_pixels(max_class_count= max_classes)

# Format results and clear old variables
unseen_mixpix, unseen_pixel_fpcs, unseen_endmembers = unseen_mixer.format_sim_results(new_results, new_f, new_endmember_indices)
del new_results, new_f, new_endmember_indices


# Inspect simulated results
print(f"{unseen_mixpix.shape[0]} pixels simulated with {unseen_mixpix.shape[1]} spectral bands \n")


In [15]:
# Add one mat class + water pixels
unseen_more_results, unseen_more_f, unseen_more_endmembers = unseen_mixer.sim_single_class_pixels(single_pixel_count= 2000, start_value = dev_pixel_count)

# Add pure water pixels
unseen_water_results_df, unseen_water_fpcs, unseen_water_endmembers = unseen_mixer.add_water_only_pixels(water_pixels_count= 5000, start_value= unseen_more_results.index[-1]+1)

# Combine with multi-class simulations
unseen_endmember_indices = pd.concat([unseen_endmembers, unseen_more_endmembers, unseen_water_endmembers], axis = 0)
unseen_sim_pix = pd.concat([unseen_mixpix, unseen_more_results.iloc[:, :-1], unseen_water_results_df], axis = 0)
unseen_pixel_fpcs = pd.concat([unseen_pixel_fpcs, unseen_more_f, unseen_water_fpcs], axis = 0)

In [16]:
# Save the unseen pixels 

np.save(sim_pix_directory + "unseen_pixels.npy", unseen_sim_pix)
np.save(sim_pix_directory + "unseen_pixel_fpcs.npy", unseen_pixel_fpcs)
np.save(sim_pix_directory + "unseen_pixel_endmembers.npy", unseen_endmember_indices)
np.save(sim_pix_directory + "data_test_indices.npy", data_test.index)

In [ ]:
# Plot the components of the simulated pixels      
plot_mixpix_components(unseen_pixel_fpcs, colours, sensor_code= sensor_code, class_code = class_code, save = save_plots, output_dir = plot_output_dir, sim_type= "unseen")

# Examine the endmembers and water spectra

In [ ]:
# De-duplicate water spectra
water_spec.drop_duplicates(inplace = True)
water_meta = water_meta.loc[water_spec.index]
print(f"Total of {water_spec.shape[0]} unique water pixels")

water_spec = water_spec/10000  # Convert to reflectance

In [24]:
if sensor_code == "psd" :
    water_spec.columns = ["Blue", "Green_i", "Green_ii", "Yellow", "Red", "Red-edge", "NIR"]


In [ ]:
# Examine the water spectra being used

x = StandardScaler().fit_transform(water_spec)

#Enmap = 12 clusters, s2 = 8 clusters, psd = 10  clusters
clusters = KMeans(n_clusters = 15, n_init ='auto').fit(x)
water_meta["cluster"] = clusters.labels_
water_spec.groupby(water_meta["cluster"]).mean().T.plot(cmap = "tab20")
plt.xticks(rotation = 45)
plt.suptitle("Mean Sentinel-2 Water Spectra by KMeans Cluster")
plt.ylabel("Reflectance")
plt.xlabel("Spectral Band")
plt.legend()
# plt.legend(title = "Cluster", loc = "best")
plt.savefig(plot_output_dir + f"{sensor_code}_water_cluster_profiles.svg", bbox_inches = "tight")
plt.show()

In [ ]:
water_meta["cluster"].value_counts().plot(kind = "bar")

# Clustering water spectra

In [ ]:
# plot kmeans clusters
plt.scatter((x[:, 1] - x[:, 2]), x[:, 4], c = water_meta["cluster"], cmap = "tab20", alpha = 0.4)
# plt.scatter((x[:, 20]-x[:, 4]), (x[:, 29]-x[:, -9])/(x[:, 29]+x[:, -9]), c = water_meta["cluster"], cmap = "tab20", alpha = 0.4)        #ENMAP 
plt.yscale("log")
plt.ylabel("Red Edge reflectance")
plt.xlabel("Normalized green and red index")

plt.title("Water clusters")
plt.show()